In [ ]:


export=False


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


def re_remove_post(x, exp = ' '):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]



In [ ]:


file_pop = path_csm / 'Pop_3 MSA ACS5_ChamberStudy2026.xlsx'
df_pop = pd.read_excel(file_pop, sheet_name='MSA')
df_pop.head()



In [ ]:


year_min = 2013
year_max = 2023

df_pop2 = df_pop.copy()
df_pop2 = df_pop2[(df_pop2['Year'] == year_max) | (df_pop2['Year'] == year_min)]
df_pop2 = df_pop2.drop(['Variable', 'Percentage', 'ME', 'Margin of Error Ratio', 'Use for Reporting?'], axis=1)

df_pop2 = df_pop2.pivot_table(index=['MSA_ID', 'MSA', 'Year'], columns='Race_Ethnicity', values='Population').reset_index()

eths = list(df_pop['Race_Ethnicity'].unique())
for eth in eths:
    col_pct_change = f'{eth}_pct_change'
    df_pop2[col_pct_change] = df_pop2[eth].pct_change()
    df_pop2.loc[df_pop2[col_pct_change] == np.inf, col_pct_change] = np.nan
    df_pop2 = df_pop2.drop(eth, axis=1)

df_pop2.loc[df_pop2['Year'] == year_min, col_pct_change] = np.nan
df_pop2 = df_pop2[df_pop2['Year'] == year_max]

df_pop2['Year'] = '2013 to 2023'

display(df_pop2)

if export:
    with pd.ExcelWriter(file_pop, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_pop2.to_excel(writer, sheet_name='PctPopGrowth', index=False)



